# Paneles solares: Faster R-CNN por fuente

Dos arquitecturas de torchvision (`fasterrcnn_mobilenet_v3_large_fpn` y `fasterrcnn_resnet50_fpn`), tres estrategias de congelamiento y los mismos tres folds que YOLO. Son 18 entrenamientos. La métrica principal es el promedio macro de mAP50-95, con el mismo peso para cada fold.

Elegí una GPU NVIDIA en *Entorno de ejecución*. En Colab, el repositorio se clona desde GitHub en el disco de la sesión y los resultados (`best.pt`, CSV, gráficos) se guardan en Google Drive, que sobrevive si la sesión se corta.

Las celdas de rutas y verificación no cargan modelos. La de entrenamiento sí.

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Fuera de Colab: no se montó Drive.")

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ana-castillo-zuain/procesamiento-de-imagenes.git"

if IN_COLAB:
    # El repo se clona en /content: son unos segundos y no hace falta subir nada a Drive.
    # /content se borra al cortarse la sesión; por eso los resultados van a Drive.
    REPO_DIR = Path("/content/procesamiento-de-imagenes")
    if REPO_DIR.is_dir():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    RESULTS_DIR = Path("/content/drive/MyDrive/paneles_resultados/RCNN")
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "RCNN" else Path.cwd()
    RESULTS_DIR = REPO_DIR / "RCNN" / "results"

CODE_DIR = REPO_DIR / "RCNN"
DATASET_DIR = REPO_DIR / "data" / "yolo"

assert CODE_DIR.joinpath("train.py").is_file(), f"No está train.py en {CODE_DIR}"
assert REPO_DIR.joinpath("YOLO", "solar_yolo").is_dir(), "Falta YOLO/solar_yolo: solar_rcnn usa sus folds y CSV."
assert DATASET_DIR.joinpath("PV01_val").is_dir(), f"No está el dataset en {DATASET_DIR}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("código    :", CODE_DIR)
print("dataset   :", DATASET_DIR)
print("resultados:", RESULTS_DIR)

In [ ]:
import torch
import torchvision

# No se instala nada: Colab ya trae torch y torchvision con CUDA.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA no está disponible. Activá una GPU en el entorno de ejecución antes de entrenar. "
        "No se cargó ningún modelo."
    )
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__, "torchvision", torchvision.__version__, "cuda", torch.version.cuda)

In [ ]:
import sys
sys.path.insert(0, str(CODE_DIR))

from solar_rcnn.config import Config
from solar_rcnn.engine import verify_setup

config = Config(dataset_dir=DATASET_DIR, results_dir=RESULTS_DIR, batch=8)
print(verify_setup(config))

## Entrenar

`None` recorre las 18 combinaciones. Para una sola, pasá una lista. Conviene empezar por la MobileNet, que es mucho más rápida, para ver que todo anda.

Si Colab se queda sin memoria, bajá `batch` en la celda anterior y volvé a correr desde ahí. Un experimento ya terminado, con su `best.pt`, no se repite. Uno cortado a la mitad sigue desde su última época.

In [ ]:
from solar_rcnn.engine import run_experiments

SELECTED_ARCHITECTURES = None  # ["fasterrcnn_mobilenet_v3_large_fpn", "fasterrcnn_resnet50_fpn"]
SELECTED_STRATEGIES = None     # ["full", "backbone", "partial"]
SELECTED_FOLDS = None          # [1, 2, 3]

outcome = run_experiments(
    config,
    architectures=SELECTED_ARCHITECTURES,
    strategies=SELECTED_STRATEGIES,
    folds=SELECTED_FOLDS,
)
print("fallos:", len(outcome["failures"]))
for failure in outcome["failures"]:
    print(failure["experiment_id"], failure["error"])

In [ ]:
import pandas as pd

pd.read_csv(RESULTS_DIR / "summary.csv")[
    ["architecture", "strategy", "fold1_map50_95", "fold2_map50_95", "fold3_map50_95", "macro_map50_95", "score_status"]
]